# Medical Predictive Modeling Pipeline

This notebook demonstrates an end-to-end predictive modeling workflow for healthcare data.

**Steps:**
1. **Data Generation**: Create synthetic patient data (demographics, vitals, labs, notes).
2. **Table Merging**: Combine disparate data sources into a single Master Analytical Table.
3. **Data Processing**: Clean, impute, scale, and split data for modeling.
4. **Model Training**: Train a Machine Learning model to predict patient risk.
5. **Lead Generation**: Identify high-risk patients for intervention.

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import uuid
import random
from datetime import datetime, timedelta
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

## 1. Data Generation
We generate synthetic medical records for 1000 patients.

In [ ]:
# Initialize Faker
fake = Faker()

# Configuration
NUM_RECORDS = 1000
OUTPUT_DIR = "medical_predictive_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mock Data Lists
DIAGNOSES = [
    ('I50.9', 'Heart Failure, Unspecified'), ('E11.9', 'Type 2 Diabetes w/o complications'),
    ('J18.9', 'Pneumonia, Unspecified'), ('N18.9', 'Chronic Kidney Disease, Unspecified'),
    ('A41.9', 'Sepsis, Unspecified'), ('I10', 'Essential Hypertension'),
    ('J44.9', 'COPD, Unspecified'), ('F32.9', 'Major Depressive Disorder'),
    ('M54.5', 'Low Back Pain'), ('K21.9', 'GERD')
]

VITALS = [
    ('Heart Rate', 'bpm', 60, 100), ('Systolic BP', 'mmHg', 90, 140),
    ('Diastolic BP', 'mmHg', 60, 90), ('SpO2', '%', 90, 100),
    ('Temperature', 'F', 97.0, 99.5), ('Respiratory Rate', 'bpm', 12, 20)
]

LABS = [
    ('WBC', 'White Blood Cell Count', 'K/uL', 4.0, 11.0),
    ('HbA1c', 'Hemoglobin A1c', '%', 4.0, 6.5),
    ('Creatinine', 'Creatinine', 'mg/dL', 0.6, 1.2),
    ('Glu', 'Glucose', 'mg/dL', 70, 100),
    ('Chol', 'Total Cholesterol', 'mg/dL', 125, 200)
]

MEDICATIONS = [
    ('Metformin', 'Oral', '500mg', 'BID'), ('Lisinopril', 'Oral', '10mg', 'Daily'),
    ('Atorvastatin', 'Oral', '20mg', 'Daily'), ('Amlodipine', 'Oral', '5mg', 'Daily'),
    ('Levothyroxine', 'Oral', '50mcg', 'Daily'), ('Albuterol', 'Inhaler', '90mcg', 'PRN'),
    ('Furosemide', 'Oral', '20mg', 'Daily'), ('Omeprazole', 'Oral', '20mg', 'Daily')
]

NOTE_TYPES = ['Progress Note', 'Discharge Summary', 'Nursing Note', 'Consult Note']
PREDICTION_MODELS = ['Sepsis_Risk_v1', 'Readmission_Risk_v2', 'Mortality_Risk_v1']

print(f"Generating {NUM_RECORDS} records...")

# 1. Patients
patients = []
patient_ids = [str(uuid.uuid4()) for _ in range(NUM_RECORDS)]

for pid in patient_ids:
    patients.append({
        'patient_id': pid,
        'mrn': f"MRN-{fake.unique.random_number(digits=8)}",
        'date_of_birth': fake.date_of_birth(minimum_age=18, maximum_age=90),
        'gender': random.choice(['M', 'F']),
        'ethnicity': random.choice(['Hispanic', 'Non-Hispanic White', 'Black/African American', 'Asian', 'Other']),
        'blood_type': random.choice(['A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-']),
        'created_at': fake.date_time_between(start_date='-5y', end_date='now')
    })
df_patients = pd.DataFrame(patients)
df_patients.to_csv(f"{OUTPUT_DIR}/patients.csv", index=False)

# 2. Encounters
encounters = []
encounter_ids = [str(uuid.uuid4()) for _ in range(NUM_RECORDS)]

for i in range(NUM_RECORDS):
    start_ts = fake.date_time_between(start_date='-1y', end_date='now')
    end_ts = start_ts + timedelta(days=random.randint(1, 14))
    
    encounters.append({
        'encounter_id': encounter_ids[i],
        'patient_id': random.choice(patient_ids),
        'encounter_type': random.choice(['Inpatient', 'Emergency', 'Outpatient']),
        'start_timestamp': start_ts,
        'end_timestamp': end_ts if random.random() > 0.1 else None,
        'discharge_disposition': random.choice(['Home', 'SNF', 'Home Health', 'Expired']) if random.random() > 0.1 else None,
        'drg_code': f"DRG-{random.randint(100, 999)}"
    })
df_encounters = pd.DataFrame(encounters)
df_encounters.to_csv(f"{OUTPUT_DIR}/encounters.csv", index=False)

# 3. Vitals & Labs (Simplified generation for notebook brevity)
vitals_data = []
labs_data = []
diag_data = []
med_data = []
pred_data = []

for _ in range(NUM_RECORDS):
    enc = random.choice(encounters)
    
    # Vitals
    vital_type, unit, min_val, max_val = random.choice(VITALS)
    val = round(random.uniform(min_val, max_val), 1)
    vitals_data.append({
        'vital_id': str(uuid.uuid4()), 'encounter_id': enc['encounter_id'],
        'vital_type': vital_type, 'vital_value': val, 'unit': unit,
        'measured_at': enc['start_timestamp'] + timedelta(minutes=random.randint(10, 600))
    })
    
    # Labs
    lab_code, test_name, unit, min_ref, max_ref = random.choice(LABS)
    val = round(random.uniform(min_ref * 0.8, max_ref * 1.2), 2)
    labs_data.append({
        'lab_id': str(uuid.uuid4()), 'encounter_id': enc['encounter_id'],
        'lab_code': lab_code, 'test_name': test_name, 'result_value': val, 'result_unit': unit,
        'reference_range': f"{min_ref} - {max_ref}",
        'collected_at': enc['start_timestamp'] + timedelta(hours=random.randint(1, 12))
    })
    
    # Diagnoses
    icd, desc = random.choice(DIAGNOSES)
    diag_data.append({
        'diagnosis_id': str(uuid.uuid4()), 'encounter_id': enc['encounter_id'],
        'icd_code': icd, 'description': desc, 'diagnosis_type': 'Primary',
        'diagnosed_at': enc['start_timestamp']
    })
    
    # Medications
    drug, route, dosage, freq = random.choice(MEDICATIONS)
    med_data.append({
        'med_id': str(uuid.uuid4()), 'encounter_id': enc['encounter_id'],
        'drug_name': drug, 'rxnorm_code': 'RX-000', 'dosage': dosage, 'route': route, 'frequency': freq,
        'start_date': enc['start_timestamp']
    })
    
    # Predictions (Target)
    risk = random.random()
    pred_data.append({
        'prediction_id': str(uuid.uuid4()), 'encounter_id': enc['encounter_id'],
        'model_name': 'Sepsis_Risk', 'risk_score': round(risk, 4),
        'actual_outcome': 1 if random.random() < risk else 0
    })

pd.DataFrame(vitals_data).to_csv(f"{OUTPUT_DIR}/vitals.csv", index=False)
pd.DataFrame(labs_data).to_csv(f"{OUTPUT_DIR}/lab_results.csv", index=False)
pd.DataFrame(diag_data).to_csv(f"{OUTPUT_DIR}/diagnoses.csv", index=False)
pd.DataFrame(med_data).to_csv(f"{OUTPUT_DIR}/medications.csv", index=False)
pd.DataFrame(pred_data).to_csv(f"{OUTPUT_DIR}/prediction_predictions.csv", index=False)

print("Data generation complete.")

## 2. Create Master Table
We merge details from Encounters, Vitals, Labs, Diagnosis, and Medications into a single row per encounter.

In [ ]:
INPUT_DIR = "medical_predictive_data"

print("Loading and Merging Data...")
patients = pd.read_csv(f"{INPUT_DIR}/patients.csv")
encounters = pd.read_csv(f"{INPUT_DIR}/encounters.csv")
vitals = pd.read_csv(f"{INPUT_DIR}/vitals.csv")
labs = pd.read_csv(f"{INPUT_DIR}/lab_results.csv")
diagnoses = pd.read_csv(f"{INPUT_DIR}/diagnoses.csv")
meds = pd.read_csv(f"{INPUT_DIR}/medications.csv")
predictions = pd.read_csv(f"{INPUT_DIR}/prediction_predictions.csv")

# 1. Base Merge
master = pd.merge(encounters, patients, on='patient_id', how='left')

# Feature Engineering: Age
master['start_timestamp'] = pd.to_datetime(master['start_timestamp'])
master['date_of_birth'] = pd.to_datetime(master['date_of_birth'])
master['age_at_encounter'] = (master['start_timestamp'] - master['date_of_birth']).dt.days // 365

# Length of Stay
master['end_timestamp'] = pd.to_datetime(master['end_timestamp'])
master['length_of_stay'] = (master['end_timestamp'] - master['start_timestamp']).dt.total_seconds() / 86400
master['length_of_stay'] = master['length_of_stay'].fillna(0)

# 2. Vitals Aggregation
vitals_agg = vitals.groupby(['encounter_id', 'vital_type'])['vital_value'].agg(['mean', 'max', 'min']).unstack()
vitals_agg.columns = [f"{col[1]}_{col[0]}" for col in vitals_agg.columns]
vitals_agg = vitals_agg.reset_index()

# 3. Labs Aggregation
labs_agg = labs.groupby(['encounter_id', 'test_name'])['result_value'].mean().unstack()
labs_agg.columns = [f"Lab_{col.replace(' ', '_')}" for col in labs_agg.columns]
labs_agg = labs_agg.reset_index()

# 4. Diagnoses Flags
comorbidities = {'Diabetes': 'E11', 'Hypertension': 'I10', 'Heart_Failure': 'I50', 'COPD': 'J44'}
diag_features = pd.DataFrame(diagnoses['encounter_id'].unique(), columns=['encounter_id'])
for name, code in comorbidities.items():
    ids = diagnoses[diagnoses['icd_code'].str.startswith(code)]['encounter_id'].unique()
    diag_features[f"Has_{name}"] = diag_features['encounter_id'].isin(ids).astype(int)

# 5. Merge Features
master = pd.merge(master, vitals_agg, on='encounter_id', how='left')
master = pd.merge(master, labs_agg, on='encounter_id', how='left')
master = pd.merge(master, diag_features, on='encounter_id', how='left')

# Fill Diagnosis NaNs with 0
for col in diag_features.columns:
    if col != 'encounter_id':
        master[col] = master[col].fillna(0).astype(int)

# 6. Add Target
targets = predictions[['encounter_id', 'actual_outcome']].drop_duplicates()
master = pd.merge(master, targets, on='encounter_id', how='left')

# Cleanup
master = master.fillna(0)
drop_cols = ['patient_id', 'mrn', 'date_of_birth', 'created_at', 'start_timestamp', 'end_timestamp', 'drg_code']
master = master.drop(columns=[c for c in drop_cols if c in master.columns])

print(f"Master Table Shape: {master.shape}")
master.head()

## 3. Data Processing & Splitting
We clean the data, scale numeric features, and split into Training and Test sets.

In [ ]:
TARGET_COL = "actual_outcome"
df = master.copy()

# Drop IDs for training
X = df.drop(columns=[TARGET_COL, 'encounter_id'])
y = df[TARGET_COL]

# Identify Features
numeric_features = [c for c in X.columns if X[c].dtype in ['int64', 'float64']]
categorical_features = [c for c in X.columns if X[c].dtype == 'object']

print(f"Numeric: {len(numeric_features)}, Categorical: {len(categorical_features)}")

# Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Get Feature Names
feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

print("Data Processed successfully.")

## 4. Model Training
Training a Random Forest Classifier to predict the outcome.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42)
rf_model.fit(X_train_df, y_train)

# Evaluate
y_pred = rf_model.predict(X_test_df)
y_probs = rf_model.predict_proba(X_test_df)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}")

## 5. Lead Generation
Identify high-risk patients who need immediate attention.

In [ ]:
leads_df = X_test_df.copy()
leads_df['Predicted_Probability'] = y_probs

def categorize_risk(prob):
    if prob >= 0.75: return 'High Priority (Immediate Action)'
    elif prob >= 0.45: return 'Medium Priority (Monitor)'
    else: return 'Low Priority'

leads_df['Risk_Category'] = leads_df['Predicted_Probability'].apply(categorize_risk)

high_priority = leads_df[leads_df['Risk_Category'].str.contains('Priority')].sort_values('Predicted_Probability', ascending=False)

print(f"Found {len(high_priority)} actionable leads.")
high_priority[['Predicted_Probability', 'Risk_Category']].head(10)